# Structured Output in LangChain

**Goal:** make an LLM return data in a predictable, typed shape (an object with named fields) instead of free-form text.

**Why it matters:** free text is hard to parse reliably. Structured output lets your code use `result.setup` directly, with no string parsing.

**Notebook flow:**
1. Set up the LLM
2. Call it normally and see the messy free-text result
3. Define a Pydantic schema
4. Use `with_structured_output(schema)` to get a typed object back

In [1]:
from langchain_openai import ChatOpenAI
import os

llm = ChatOpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    # api_key=os.getenv("OPENAI_API_KEY"),
    model="nvidia/nemotron-3.5-lightning-30b-a3b",
)

## 1. Setting up the LLM

- `ChatOpenAI` is LangChain's chat model wrapper for any **OpenAI-compatible API**.
- `base_url` points it at NVIDIA's endpoint instead of OpenAI, so we can use NVIDIA-hosted models with the same class.
- `model` is the model name on that provider.
- The API key is read from the `OPENAI_API_KEY` environment variable by default (the commented `api_key=` line shows the explicit way).

In [2]:
result = llm.invoke("tell me a joke and generate the output in a key value pair.")
result.content

'joke: "Why don\'t scientists trust atoms? Because they make up everything!"'

### What happened here?

- `llm.invoke(prompt)` sends the prompt and returns a message object; `.content` is the text.
- We *asked* for a key-value pair in the prompt, and got a **plain string** like `joke: "..."`.
- Problem: this is only a string. The format depends on the model's mood, and you'd have to parse it yourself. Not reliable.

## 2. Structured output using Pydantic

Instead of begging the model in the prompt, we give it a **schema**: a Pydantic class describing the fields we want.

- `BaseModel`: Pydantic base class, gives validation and typed objects.
- `Field(description=...)`: the description is sent to the model, so it knows what each field should contain. Write good descriptions, they act like instructions.

In [ ]:
from pydantic import BaseModel,Field

class llm_schema(BaseModel):
    setup: str = Field(description="The setup for the joke")
    punchline: str = Field(description="The punchline for the joke")


In [5]:
llm_structured_output = llm.with_structured_output(llm_schema)
llm_structured_output.invoke("Tell me a joke")

llm_schema(setup="Why don't scientists trust atoms?", punchline='Because they make up everything!')

### What happened here?

- `llm.with_structured_output(llm_schema)` returns a **new runnable** wrapping the LLM. Under the hood LangChain uses tool/function calling (or JSON mode) to force the model to fill the schema.
- `.invoke("Tell me a joke")` now returns an `llm_schema` **object**, not a string.
- Access fields directly: `result.setup`, `result.punchline`.
- The prompt no longer needs to say "give me setup and punchline in key-value form"; the schema does that job.

**Key takeaway:** `with_structured_output(PydanticClass)` = prompt in, validated typed object out.

**Tip:** you can also pass a `TypedDict` or a JSON schema dict; Pydantic is preferred because it validates the output.